# Practice Activity: Weather as a Business Covariate

**GSB 5544 · Computing and Machine Learning for Business Analytics**

You manage inventory planning for a beverage distributor with two markets: **San Luis Obispo, CA** and **Phoenix, AZ**. Before anyone builds a demand model, you need each market's weather history, and it lives in a bucket that is far larger than your laptop.

Everything you need is in the pre-class reading. Each question has an empty code cell; some also ask for a short written answer in a markdown cell.

**Stations**

| Market | Station ID | Name |
|---|---|---|
| San Luis Obispo | `USC00047851` | SAN LUIS OBISPO POLY, CA |
| Phoenix | `USW00023183` | PHOENIX AIRPORT, AZ |

Run the setup cell first.

In [ ]:
import io
import time
import psutil
import boto3
import pandas as pd
import matplotlib.pyplot as plt
from botocore import UNSIGNED
from botocore.config import Config

BUCKET = "noaa-ghcn-pds"
COLS = ["id", "date", "element", "value", "m_flag", "q_flag", "s_flag", "obs_time"]
STATIONS = {"San Luis Obispo": "USC00047851", "Phoenix": "USW00023183"}

s3 = boto3.client("s3", region_name="us-east-1", config=Config(signature_version=UNSIGNED))


def human(n, base=1000):
    for unit in ["B", "KB", "MB", "GB", "TB", "PB"]:
        if n < base:
            return f"{n:,.1f} {unit}"
        n /= base
    return f"{n:,.1f} EB"

## Set 1 · Capacity

### Q1. Your machine
Report physical cores, total RAM, available RAM, and free disk in GB. Store available RAM in bytes as `MY_RAM`.

In [ ]:
vm = psutil.virtual_memory()
du = psutil.disk_usage("/")
MY_RAM = vm.available

print(f"Physical cores : {psutil.cpu_count(logical=False)}")
print(f"Total RAM      : {vm.total / 1e9:.1f} GB")
print(f"Available RAM  : {vm.available / 1e9:.1f} GB")
print(f"Free disk      : {du.free / 1e9:.1f} GB")

### Q2. Rows that fit
A GHCN row is roughly 50 bytes on disk, and a text-heavy CSV needs about 3× its disk size once loaded in pandas. How many raw rows could your available RAM hold? Print the number with thousands separators.

In [ ]:
BYTES_PER_ROW = 50
MULT = 3
rows_fit = MY_RAM // (BYTES_PER_ROW * MULT)
print(f"About {rows_fit:,} rows fit in {human(MY_RAM)} of available RAM")

### Q3. A decade of weather
List the by-year files for 2016 through 2025 and compute their **total** size. Print the total in readable units, then print how many copies of your available RAM that is.

*Hint: a `Prefix` of `csv/by_year/201` returns 2010–2019; you will need two listings or a filter.*

In [ ]:
total = 0
for prefix in ["csv/by_year/201", "csv/by_year/202"]:
    resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix)
    for obj in resp["Contents"]:
        year = int(obj["Key"].split("/")[-1][:4])
        if 2016 <= year <= 2025:
            total += obj["Size"]
            print(f"{obj['Key']:<28} {human(obj['Size'])}")

print()
print("Total 2016–2025 :", human(total))
print("Copies of my RAM:", f"{total / MY_RAM:.1f}×")

## Set 2 · Vocabulary in action

### Q4. What is under `csv/`?
List the common prefixes directly under `csv/` in the bucket. Then, in the markdown cell that follows, explain in one or two sentences why `by_year/` is not a folder and what S3 is actually doing when it shows you one.

In [ ]:
resp = s3.list_objects_v2(Bucket=BUCKET, Prefix="csv/", Delimiter="/")
print([p["Prefix"] for p in resp.get("CommonPrefixes", [])])

*Your answer:*

### Q5. Metadata only
Using a single call that does **not** download the file, print the size (readable) and the last-modified date of `csv/by_year/2024.csv`. Then print whether a plain `pd.read_csv` of it would fit in your RAM at 3×.

In [ ]:
meta = s3.head_object(Bucket=BUCKET, Key="csv/by_year/2024.csv")
size = meta["ContentLength"]
print("Size         :", human(size))
print("Last modified:", meta["LastModified"])
print("Fits at 3×?  :", size * 3 < MY_RAM)

## Set 3 · Extraction

### Q6. Pull both markets
Write a function `load_station(station_id)` that:

1. prints the object's size before reading it,
2. checks whether the file has a header row using a range request,
3. reads the file keeping only `id`, `date`, `element`, `value`,
4. converts `date` to a datetime,
5. keeps only `TMAX` and `PRCP` rows from **2015 onward**,
6. returns the DataFrame.

Call it for both stations and combine the results into one DataFrame called `wx` with a `market` column.

In [ ]:
def load_station(station_id):
    key = f"csv/by_station/{station_id}.csv"
    print(station_id, "size:", human(s3.head_object(Bucket=BUCKET, Key=key)["ContentLength"]))

    first = s3.get_object(Bucket=BUCKET, Key=key, Range="bytes=0-99")["Body"].read().decode()
    has_header = first.upper().startswith("ID,")

    raw = s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()
    df = pd.read_csv(
        io.BytesIO(raw),
        header=0 if has_header else None,
        names=COLS,
        usecols=["id", "date", "element", "value"],
        dtype={"id": "string", "element": "string"},
    )
    df["date"] = pd.to_datetime(df["date"].astype(str), format="%Y%m%d")
    df = df[df["element"].isin(["TMAX", "PRCP"]) & (df["date"] >= "2015-01-01")]
    return df.reset_index(drop=True)


frames = []
for market, sid in STATIONS.items():
    d = load_station(sid)
    d["market"] = market
    frames.append(d)

wx = pd.concat(frames, ignore_index=True)
print(wx.shape)
wx.groupby(["market", "element"]).size()

### Q7. Hot days per year
Temperatures are stored in tenths of a degree Celsius. For each market and year, count the number of days with a daily maximum of **35 °C or higher** (95 °F). Produce a table with years as rows and markets as columns.

In [ ]:
tmax = wx[wx["element"] == "TMAX"].copy()
tmax["tmax_c"] = tmax["value"] / 10
tmax["year"] = tmax["date"].dt.year

hot = (
    tmax[tmax["tmax_c"] >= 35]
    .groupby(["year", "market"]).size()
    .unstack("market")
    .fillna(0).astype(int)
)
hot

### Q8. Monthly rainfall
Precipitation is stored in tenths of a millimeter. For each market, compute total precipitation per month in millimeters, then show the **average for each calendar month** across all years (so 12 rows per market). Which month is wettest in each market?

In [ ]:
prcp = wx[wx["element"] == "PRCP"].copy()
prcp["prcp_mm"] = prcp["value"] / 10

monthly_total = (
    prcp.set_index("date")
        .groupby("market")["prcp_mm"]
        .resample("ME").sum()           # "M" on pandas < 2.2
        .reset_index()
)
monthly_total["month"] = monthly_total["date"].dt.month

climatology = monthly_total.groupby(["market", "month"])["prcp_mm"].mean().unstack("market").round(1)
print(climatology)
print()
print("Wettest month:", climatology.idxmax().to_dict())

## Set 4 · Visualize and interpret

### Q9. One chart, two markets
Plot hot days per year (from Q7) for both markets on a single line chart. Label the axes with units and give the chart a title that says what it shows.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for market in hot.columns:
    ax.plot(hot.index, hot[market], marker="o", linewidth=1.3, label=market)
ax.set_xlabel("Year")
ax.set_ylabel("Days with max temperature ≥ 35 °C")
ax.set_title("Hot days per year, 2015 onward")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

### Q10. The business read
In four to six sentences, answer as the inventory planner:

- What does the chart imply about how differently the two markets should be stocked across the year?
- Name **two** other datasets you would want to join to this weather history before trusting a demand model, and say where each one probably lives (your laptop, a company database, or a cloud bucket).

*Your answer:*

### Q11. Stretch: the wrong way, on purpose
Extract the **same Phoenix rows for 2024** by reading `csv/by_year/2024.csv` in chunks of one million rows, filtering each chunk to the Phoenix station. Time it. Then compare the row count to what you get by filtering `wx` to Phoenix and 2024.

In the markdown cell after, explain which extraction is better and why, using the phrase "filter at the source."

*This reads 1.3 GB over the network. Expect several minutes. Skip if your connection is slow.*

In [ ]:
key = "csv/by_year/2024.csv"
first = s3.get_object(Bucket=BUCKET, Key=key, Range="bytes=0-99")["Body"].read().decode()
has_header = first.upper().startswith("ID,")

t0 = time.perf_counter()
chunks = pd.read_csv(
    f"s3://{BUCKET}/{key}",
    storage_options={"anon": True},
    header=0 if has_header else None,
    names=COLS,
    usecols=["id", "date", "element", "value"],
    dtype={"id": "string", "element": "string"},
    chunksize=1_000_000,
)
by_year = pd.concat(c[c["id"] == STATIONS["Phoenix"]] for c in chunks)
by_year = by_year[by_year["element"].isin(["TMAX", "PRCP"])]
elapsed = time.perf_counter() - t0

by_station = wx[(wx["market"] == "Phoenix") & (wx["date"].dt.year == 2024)]

print(f"by_year   : {len(by_year):,} rows in {elapsed:,.0f} s (read {human(s3.head_object(Bucket=BUCKET, Key=key)['ContentLength'])})")
print(f"by_station: {len(by_station):,} rows (read {human(s3.head_object(Bucket=BUCKET, Key='csv/by_station/USW00023183.csv')['ContentLength'])})")

*Your answer:*

---

**Submit** this notebook with all cells run. Your written answers for Q4, Q10, and Q11 go in the markdown cells provided.